In [1]:
import sys
from pathlib import Path

# Find project root dynamically
def get_project_root() -> Path:
    try:
        path = Path(__file__).resolve()
        for parent in [path] + list(path.parents):
            if (parent / "requirements.txt").exists() or (parent / "project").exists():
                return parent
    except NameError:
        pass
    path = Path.cwd().resolve()
    for parent in [path] + list(path.parents):
        if (parent / "requirements.txt").exists() or (parent / "project").exists():
            return parent
    return path

ROOT = get_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import torch.nn as nn
from torch.nn import functional as F

# parameters
batch_size = 32       # 32 sequences at once
block_size = 8        # length of each seq = 8
steps = 5000          # 3000 training updates
learning_rate = 1e-3  # size of parameter updates
n_embd = 32           # no. of features associated with each token
head_size = 32

# importing dataset
with open(ROOT / 'TASK 1' / 'input.txt', 'r', encoding='utf-8') as f:
    text = f.read()     # saves as one python string

# creating vocabulary
chars = sorted(list(set(text)))     # gets unique characters and sorts them
vocab_size = len(chars)             # vocab_size = total no. of unique characters

# dictionaries
stoi = {ch: i for i, ch in enumerate(chars)} # enumerate assigns a number
itos = {i: ch for i, ch in enumerate(chars)}

# encoder - decoder
def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

# encode entire dataset
data = torch.tensor(encode(text), dtype=torch.long)

# train - test split
n = int(0.9 * len(data))

train_data = data[:n]     # 90%
test_data = data[n:]      # 10%

# batching : goal - creating training examples

def get_batch(split):

    data = train_data if split == 'train' else test_data

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])

    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x, y



class Head(nn.Module): # one self-attention unit

    def __init__(self, n_embd, head_size, block_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        # this creates a lower triangular mask filled with 1's in tri and 0 elsewhere
        # this prevents future tokens from being visible

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1)     # (B, T, T)
        wei = wei * (k.size(-1) ** -0.5)  # scale by 1/sqrt(d_k)

        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        wei = F.softmax(wei, dim=-1) # (B, T, T)

        # weighted sum of values
        out = wei @ v # (B, T, head_size)
        return out

# model

class BigramLanguageModel(nn.Module):

    # some syntax i didnt know about so had to gpt
    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )

        # transformer should know order of characters too so positional embedding added
        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        # self attention head
        self.sa_head = Head(
            n_embd,
            head_size,
            block_size
        )

        # linear layer
        self.lm_head = nn.Linear(
            head_size,
            vocab_size
        )
        # attention outputs (B,T,head_size) but for predictions we need (B,T,vocab_size)
        # so linear layer will convert features → vocabulary logits

    # forward pass
    def forward(self, idx, targets=None):

        # idx: (B,T)
        B, T = idx.shape
        token_emb = self.token_embedding_table(idx)
        # token_emb: (B,T,n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        # pos_emb: (T,n_embd)
        # PyTorch automatically copies position embeddings across batch dimension.
        # So every sequence gets same positional vectors.

        x = token_emb + pos_emb
        # before a character had same embedding everywhere
        # but now the same character at two diff positions have diff embeddings

        x = self.sa_head(x)

        logits = self.lm_head(x)

        if targets is None:
            loss = None

        else:
            B, T, C = logits.shape

            logits = logits.view(B * T, C) # flatteing as entropy wants (N,C) and (N)
            targets = targets.view(B * T)  # here N = B*T

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            # gets predictions
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)

            # focus on last timestep
            logits = logits[:, -1, :]

            # logits convert to probabilities
            probs = F.softmax(logits, dim=-1)

            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)

            # append sampled token
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

# initialize model
model = BigramLanguageModel(vocab_size)

# optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# training loop
for iter in range(steps):

    # sampling batch
    xb, yb = get_batch('train')

    # forward pass
    logits, loss = model(xb, yb)

    # clear gradients
    optimizer.zero_grad(set_to_none=True)

    # backprop
    loss.backward()

    # update parameters
    optimizer.step()

    # print loss
    if iter % 300 == 0:
        print(f"step {iter}: loss {loss.item():.4f}")

# generate text
context = torch.zeros((1, 1), dtype=torch.long)

generated_tokens = model.generate(
    context,
    max_new_tokens=500
)

generated_text = decode(
    generated_tokens[0].tolist()
)

print("\nGenerated Text:\n")
print(generated_text)


step 0: loss 4.2309
step 300: loss 2.9241
step 600: loss 2.7000
step 900: loss 2.5525
step 1200: loss 2.3864
step 1500: loss 2.5024
step 1800: loss 2.5780
step 2100: loss 2.4771
step 2400: loss 2.4320
step 2700: loss 2.5805
step 3000: loss 2.5067
step 3300: loss 2.4384
step 3600: loss 2.2170
step 3900: loss 2.3974
step 4200: loss 2.3338
step 4500: loss 2.5487
step 4800: loss 2.3669

Generated Text:


ENShan,
I'st sithy; ha Rlalllo stheravione hin ngel lge.

And tenshe nd o'thetoevierd wes ay, the I he thkiterwo te tavacrem ve fo ove! pornecor'ss ard mu wheasn, st.

INELTETCowurfo LAoth kncru.

ud,
Aw tthil ave sarid fo now nous ss, of waty the and harde sind ly ty ty py growr n's hof fir me.

OLI pelofout deet ine ghe ok cuan op.
Wes.
Bu froon dithe.
And,
HHre by ty, on an wat fiot scharnd ivuer we nefarisan

AdBeaf blovero'd ywime illif bet, thave he lgine.

POLORA:
BEVE:
To barn yot,
Whik 
